# LlamaIndex

> A self-contained refresher. LlamaIndex is the **data framework** that wires your
> private documents into an LLM — it owns the RAG pipeline so you don't hand-roll
> chunking, vector plumbing, and prompt assembly.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** LlamaIndex (formerly *GPT Index*) is a Python/TS framework for building
LLM apps over your own data. Its center of gravity is **retrieval-augmented generation
(RAG)**: take a pile of documents, turn them into a queryable index, and answer questions
by retrieving the relevant bits and feeding them to an LLM.

**The problem it solves.** A bare LLM only knows its training data and whatever fits in
the prompt. To ground answers in *your* PDFs, wikis, or DB rows you need an ingestion →
chunking → embedding → indexing → retrieval → synthesis pipeline. Writing that by hand is
a lot of fiddly glue (see [`rag.ipynb`](rag.ipynb) for the from-scratch version).
LlamaIndex gives you that pipeline as a few composable objects with sane defaults and
~300 data loaders + vector-store integrations.

**When to reach for it.**
- You have unstructured data and want question-answering / chat grounded in it.
- You want to swap embedding models, vector stores, or response strategies without rewrites.
- You want batteries-included loaders (Notion, Slack, SQL, web, PDFs) via LlamaHub.

**When *not* to.**
- A single LLM call with everything in context — just use the model SDK.
- You need a broad agent/tool-orchestration framework first and RAG second (LangChain is
  wider; though LlamaIndex has agents/workflows too — see *LlamaIndex Agents*).
- You only need a vector store — Chroma/FAISS/Pinecone alone may be enough.

## 2. Mental Model

Think **"a database you query in natural language."** LlamaIndex is ETL + a query layer
for unstructured data. The pipeline has two halves:

```
INDEXING (offline, build once)            QUERYING (online, per question)
  Documents                                 Query string
     │ node parser / splitter                  │ embed query
     ▼                                          ▼
  Nodes (chunks + metadata)               Retriever  ──► top-k Nodes
     │ embedding model                          │
     ▼                                          ▼
  Embeddings ──► Index (vector store)     Response Synthesizer
                                            (stuff Nodes into an LLM prompt)
                                                  │
                                                  ▼
                                              Response
```

A **query engine** is just `retriever + response synthesizer` bundled together. Build the
index once; query it many times. Everything downstream of ingestion operates on **Nodes**,
not raw Documents.

## 3. Key Concepts

| Concept | What to hold in your head |
|---|---|
| **Document** | A source blob (text) + metadata. The raw input unit. |
| **Node** | A chunk of a Document plus relationships (prev/next/source). Everything downstream operates on Nodes. |
| **Node parser / splitter** | Turns Documents → Nodes. `SentenceSplitter(chunk_size, chunk_overlap)` is the default. |
| **Embedding model** | Maps text → vector. Set via `Settings.embed_model`. Default is OpenAI. |
| **LLM** | The generation model. Set via `Settings.llm`. Default is OpenAI. |
| **Index** | `VectorStoreIndex` (semantic, most common), `SummaryIndex`, `KeywordTableIndex`, `PropertyGraphIndex`. |
| **Retriever** | Pulls the top-k relevant Nodes (`similarity_top_k`). `index.as_retriever()`. |
| **Response synthesizer** | Folds retrieved Nodes into the final prompt. Modes: `compact` (default), `refine`, `tree_summarize`. |
| **Query engine** | High-level `index.as_query_engine()` = retriever + synthesizer. |
| **Settings** | Global config for `llm`, `embed_model`, `node_parser` (replaced the old `ServiceContext`). |
| **StorageContext** | Persist/load index + docstore + vector store to disk so you don't re-embed. |

The whole API is: configure `Settings`, build an `Index`, call `as_query_engine()`.

## 4. Setup

```bash
# Core framework — enough for local/mock components, no model SDKs pulled in.
pip install llama-index-core

# The convenience meta-package wires in OpenAI defaults (needs OPENAI_API_KEY):
pip install llama-index

# Pick your providers à la carte (the v0.10+ way):
pip install llama-index-llms-anthropic llama-index-embeddings-huggingface
```

**Heads-up:** since **v0.10** LlamaIndex is a monorepo — `llama-index-core` plus separate
integration packages. Old `from llama_index import ...` imports moved to
`from llama_index.core import ...`. The **default LLM and embedding model are OpenAI**, so
out of the box you need `OPENAI_API_KEY`. Below we sidestep that entirely with mock/local
components so the notebook runs on CPU with no keys and no downloads.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no API key needed).
import importlib.util
import os
import sys


def have(mod: str) -> str:
    try:
        return "installed" if importlib.util.find_spec(mod) else "not installed"
    except ModuleNotFoundError:
        return "not installed"  # a parent package is missing


print(f"python                 : {sys.version.split()[0]}")
print(f"numpy                  : {have('numpy')}")
print(f"llama-index-core       : {have('llama_index')}")
print(f"llama-index-anthropic  : {have('llama_index.llms.anthropic')}")
print(f"OPENAI_API_KEY         : {'set' if os.getenv('OPENAI_API_KEY') else 'not set'}")
print(f"ANTHROPIC_API_KEY      : {'set' if os.getenv('ANTHROPIC_API_KEY') else 'not set'}")

python                 : 3.13.7
numpy                  : installed
llama-index-core       : installed
llama-index-anthropic  : not installed
OPENAI_API_KEY         : not set
ANTHROPIC_API_KEY      : not set


## 5. Worked Examples

We build the same tiny RAG pipeline twice. **Example 1** does it in ~15 lines of NumPy so
you can *see* every step LlamaIndex performs. **Example 2** runs the *real* LlamaIndex API
with a deterministic CPU embedding and a `MockLLM`, so it executes offline with no keys.
**Example 3** shows how to swap in a real Anthropic LLM, gated behind an API-key check.

In [2]:
# Example 1 — the RAG pipeline by hand, so the LlamaIndex objects map onto something.
# Document -> Node (chunk) -> embedding -> index -> retrieve top-k.
import hashlib

import numpy as np

DIM = 64


def embed(text: str) -> np.ndarray:
    """Deterministic bag-of-words hashing embedding — no model, no network."""
    v = np.zeros(DIM)
    for tok in text.lower().split():
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n else v


documents = [
    "LlamaIndex converts documents into nodes, embeds them, and stores them in an index.",
    "A retriever fetches the top-k most similar nodes for a given query.",
    "A response synthesizer stuffs the retrieved nodes into a prompt for the LLM.",
    "Flash attention is a fast GPU kernel for transformer attention, unrelated to retrieval.",
]

# "Index" = the matrix of node embeddings.
index = np.vstack([embed(d) for d in documents])


def retrieve(query: str, top_k: int = 2):
    scores = index @ embed(query)  # cosine sim (vectors are unit-norm)
    order = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), documents[i]) for i in order]


for score, text in retrieve("what does the retriever do?"):
    print(f"{score:.3f}  {text}")

0.239  A retriever fetches the top-k most similar nodes for a given query.
0.217  A response synthesizer stuffs the retrieved nodes into a prompt for the LLM.


In [3]:
# Example 2 — the SAME pipeline with the real LlamaIndex API, fully offline.
# MockLLM + a deterministic embedding => no API keys, no downloads.
from importlib.util import find_spec

if find_spec("llama_index") is None:
    print("llama-index-core not installed — `pip install llama-index-core` to run this cell.")
else:
    from typing import List

    from llama_index.core import Document, Settings, VectorStoreIndex
    from llama_index.core.embeddings import BaseEmbedding
    from llama_index.core.llms import MockLLM
    from llama_index.core.node_parser import SentenceSplitter

    class HashEmbedding(BaseEmbedding):
        """Reuse Example 1's hashing embedding through LlamaIndex's interface."""

        def _get_text_embedding(self, text: str) -> List[float]:
            return embed(text).tolist()

        def _get_query_embedding(self, query: str) -> List[float]:
            return embed(query).tolist()

        async def _aget_query_embedding(self, query: str) -> List[float]:
            return embed(query).tolist()

    # Global defaults — this is how you avoid the OpenAI defaults.
    Settings.embed_model = HashEmbedding()
    Settings.llm = MockLLM()  # returns a stub answer; swap for a real LLM in Example 3
    # tokenizer=str.split keeps SentenceSplitter from downloading tiktoken's vocab.
    Settings.node_parser = SentenceSplitter(chunk_size=64, chunk_overlap=8, tokenizer=str.split)

    docs = [Document(text=d) for d in documents]
    vindex = VectorStoreIndex.from_documents(docs)

    retriever = vindex.as_retriever(similarity_top_k=2)
    print("Retrieved nodes (real LlamaIndex retriever):")
    for node in retriever.retrieve("what does the retriever do?"):
        print(f"  {node.score:.3f}  {node.get_content()[:60]}")

    # A query engine = retriever + response synthesizer. With MockLLM the answer is a
    # placeholder, but the wiring (and the assembled prompt) is exactly real.
    engine = vindex.as_query_engine()
    response = engine.query("what does the retriever do?")
    print(f"\nquery engine returned {len(response.source_nodes)} source nodes; "
          f"response type = {type(response).__name__}")

Retrieved nodes (real LlamaIndex retriever):
  0.239  A retriever fetches the top-k most similar nodes for a given
  0.217  A response synthesizer stuffs the retrieved nodes into a pro

query engine returned 2 source nodes; response type = Response


In [4]:
# Example 3 — swap MockLLM for a real Anthropic model. Gated so the notebook still runs.
import os

try:
    from llama_index.llms.anthropic import Anthropic  # pip install llama-index-llms-anthropic

    have_anthropic = True
except ImportError:
    have_anthropic = False

if os.getenv("ANTHROPIC_API_KEY") and have_anthropic:
    from llama_index.core import Settings

    Settings.llm = Anthropic(model="claude-haiku-4-5")  # cheap model is plenty for RAG synthesis
    engine = vindex.as_query_engine()  # reuse the index built in Example 2
    print(engine.query("In one sentence, what does the retriever do?"))
else:
    print("[no ANTHROPIC_API_KEY / llama-index-llms-anthropic — showing the call shape]\n")
    print("from llama_index.llms.anthropic import Anthropic")
    print("Settings.llm = Anthropic(model='claude-haiku-4-5')")
    print("engine = vindex.as_query_engine()")
    print("engine.query('In one sentence, what does the retriever do?')")
    print("\n# Retrieval still runs locally; only the final synthesis call hits the API.")

[no ANTHROPIC_API_KEY / llama-index-llms-anthropic — showing the call shape]

from llama_index.llms.anthropic import Anthropic
Settings.llm = Anthropic(model='claude-haiku-4-5')
engine = vindex.as_query_engine()
engine.query('In one sentence, what does the retriever do?')

# Retrieval still runs locally; only the final synthesis call hits the API.


## 6. Gotchas & Pitfalls

- **Silent OpenAI defaults.** Out of the box `Settings.llm`/`embed_model` are OpenAI, so a
  fresh `VectorStoreIndex.from_documents(...)` fails with an auth error (or quietly bills you)
  unless you set them. Always configure `Settings` explicitly.
- **The v0.10 import split.** `from llama_index import VectorStoreIndex` no longer works —
  it's `from llama_index.core import VectorStoreIndex`. Integrations live in their own
  packages (`llama-index-llms-anthropic`, `llama-index-vector-stores-chroma`, …). And
  `ServiceContext` is gone — use `Settings`.
- **tiktoken/nltk downloads.** `SentenceSplitter` tokenizes with tiktoken by default and
  hits the network on first use; sentence splitting may pull NLTK data. Pass a `tokenizer`
  (e.g. `str.split`) or pre-download for offline/CI runs.
- **Chunking is the lever.** Chunks too big dilute retrieval and waste context; too small
  lose meaning. Tune `chunk_size`/`chunk_overlap` — it matters more than the index type.
- **`similarity_top_k` trade-off.** Too low misses context; too high floods the prompt with
  noise and cost. Pair a generous retrieve with a reranker (see [`rerankers.ipynb`](rerankers.ipynb)).
- **Re-embedding cost.** Rebuilding the index re-embeds every node. Persist with
  `StorageContext` / `index.storage_context.persist(...)` and reload instead.
- **Every query is three calls.** embed-the-query → vector search → LLM synthesis. Latency
  and cost add up; cache, batch, and pick a cheap synthesis model.

## 7. When to Use vs Alternatives

| Option | Use it when | Trade-off vs LlamaIndex |
|---|---|---|
| **Hand-rolled RAG** ([`rag.ipynb`](rag.ipynb)) | You want full control / minimal deps, or to learn the mechanics. | More code, but no abstraction to fight; you own every knob. |
| **LangChain** | You need broad agent/tool/chain orchestration first, RAG second. | Wider but shallower on data; many teams use LlamaIndex *for retrieval inside* a LangChain app. |
| **Haystack** | You want a production pipeline/graph DAG with strong eval tooling. | More pipeline-/ops-oriented; LlamaIndex is faster to a first query engine. |
| **A vector DB alone** (Chroma/FAISS/Pinecone) | You only need store + similarity search. | Those *store and retrieve* vectors; LlamaIndex orchestrates chunking + retrieval + synthesis on top (see [`chromadb.ipynb`](chromadb.ipynb), [`faiss.ipynb`](faiss.ipynb), [`pinecone.ipynb`](pinecone.ipynb)). |

**Rule of thumb:** LlamaIndex is the sweet spot when the job is *"answer questions over my
documents"* and you want the RAG plumbing handled but still swappable. Drop down to a raw
vector DB when you only need search; reach for LangChain/agents when retrieval is one tool
among many.

## 8. Resources

- **Official docs** — https://docs.llamaindex.ai/
- **Starter tutorial (5-line RAG)** — https://docs.llamaindex.ai/en/stable/getting_started/starter_example/
- **High-level concepts** — https://docs.llamaindex.ai/en/stable/getting_started/concepts/
- **GitHub (run-llama/llama_index)** — https://github.com/run-llama/llama_index
- **LlamaHub** (loaders, tools, vector-store integrations) — https://llamahub.ai/

**Related notebooks:** [`rag.ipynb`](rag.ipynb) · [`rerankers.ipynb`](rerankers.ipynb) ·
[`chromadb.ipynb`](chromadb.ipynb) · [`vector-embeddings.ipynb`](vector-embeddings.ipynb) ·
*LlamaIndex Agents* (next topic in this domain).